 Importing Libraries

In [47]:
import pandas as pd
import numpy as np
from prophet import Prophet
from tqdm import tqdm
from functools import reduce
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import VotingRegressor
import joblib

Loading the Data

In [33]:
df=pd.read_csv("../ISI_dataset/merged_mustard_reservoir.csv")
df.head()

,state_name,crop_name,apy_item_interval_start,temperature_recorded_date,state_temperature_max_val,state_temperature_min_val,state_rainfall_val,yield,FRL,Live Cap FRL,Level,Current Live Storage
0,Andhra Pradesh,rapeseed &mustard,2000,2000-01-01,30.38,14.47,0.0,0.23394,152.296667,2.838333,266.30,6.390
1,Andhra Pradesh,rapeseed &mustard,2000,2000-01-02,30.04,13.96,0.0,0.23394,152.296667,2.838333,266.18,6.330
2,Andhra Pradesh,rapeseed &mustard,2000,2000-01-03,29.92,12.98,0.0,0.23394,152.296667,2.838333,266.09,6.286
3,Andhra Pradesh,rapeseed &mustard,2000,2000-01-04,29.98,12.23,0.0,0.23394,152.296667,2.838333,266.03,6.257
4,Andhra Pradesh,rapeseed &mustard,2000,2000-01-05,29.77,13.24,0.0,0.23394,152.296667,2.838333,265.97,6.228


Changing type of date to datetime and renaming year

In [34]:
df['temperature_recorded_date'] = pd.to_datetime(df['temperature_recorded_date'])
df['year'] = df['temperature_recorded_date'].dt.year

Droping unreliable states because there is not enough years of data

In [35]:
# Use only data till 2022 for training
df = df[df['year'] < 2023].copy()

# Drop unreliable states
df = df[~df['state_name'].isin(['Jharkhand', 'Uttarakhand'])]

# Group annually to match 2023 structure
df_annual = df.groupby(['state_name', 'crop_name', 'year']).agg({
    'state_rainfall_val': 'sum',
    'state_temperature_max_val': 'mean',
    'state_temperature_min_val': 'mean',
    'Live Cap FRL': 'mean',
    'FRL': 'mean',
    'Level': 'mean',
    'Current Live Storage': 'mean',
    'yield': 'mean'
}).reset_index()

In [36]:
df_annual.head()

,state_name,crop_name,year,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,yield
0,Andhra Pradesh,rapeseed &mustard,2000,942.03,34.738197,18.885082,2.838333,152.296667,179.065178,2.636298,0.23394
1,Andhra Pradesh,rapeseed &mustard,2001,918.85,35.137890,18.995397,2.838333,152.296667,171.736301,1.820863,0.32201
2,Andhra Pradesh,rapeseed &mustard,2002,654.57,35.292247,18.858493,2.838333,152.296667,169.014014,1.402067,0.21904
3,Andhra Pradesh,rapeseed &mustard,2003,832.53,35.550356,19.248548,2.838333,152.296667,161.787795,0.781289,0.25460
4,Andhra Pradesh,rapeseed &mustard,2004,786.89,34.954836,18.399536,2.838333,152.296667,164.511298,1.314224,0.24951


In [37]:
# One-hot encode 'state_name'
df_encoded = pd.get_dummies(df_annual, columns=['state_name'])

# Define features: original + one-hot encoded state columns
state_columns = [col for col in df_encoded.columns if col.startswith('state_name_')]

Train test split (till 2020 is taken as training data and years 2021 & 2022 are in testing data)

In [38]:
# Define features and target
features = ['state_rainfall_val', 'state_temperature_max_val', 'state_temperature_min_val', 'Live Cap FRL', 'FRL','Level','Current Live Storage']+ state_columns

# Split manually by year
train_df = df_encoded[df_encoded['year'] <= 2020]
test_df = df_encoded[df_encoded['year'].between(2021, 2022)]

In [39]:
X_train = train_df[features]
y_train = train_df['yield']
X_test = test_df[features]
y_test = test_df['yield']

In [40]:
# Models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR()
}

# Results container
results = []

# Loop through models
for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    results.append({
        'Model': name,
        'Train R²': round(train_r2, 4),
        'Test R²': round(test_r2, 4),
        'Train RMSE': round(train_rmse, 2),
        'Test RMSE': round(test_rmse, 2)
    })

# Display results
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='Test R²', ascending=False))

                      Model  Train R²  Test R²  Train RMSE  Test RMSE
2                   XGBoost    1.0000   0.8474        0.00       0.22
1             Random Forest    0.9554   0.8023        0.11       0.25
3         Gradient Boosting    0.9756   0.7990        0.08       0.25
0         Linear Regression    0.6840   0.7795        0.30       0.26
4  Support Vector Regressor    0.3417   0.1628        0.43       0.51


Incoperating our top 2 or 3 best perfoming models

In [48]:
# Initialize individual models
gb = GradientBoostingRegressor(random_state=42)
rf = RandomForestRegressor()
xgb = XGBRegressor(random_state=42)

# Ensemble model
ensemble = VotingRegressor(estimators=[
    ('gb', gb),
    ('rf', rf),
    ('xgb', xgb)
])

# Fit ensemble
ensemble.fit(X_train, y_train)

# Predict
train_pred = ensemble.predict(X_train)
test_pred = ensemble.predict(X_test)

# Evaluate
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("📊 Ensemble Performance:")
print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")


📊 Ensemble Performance:
Train R²: 0.9865, Test R²: 0.8554
Train RMSE: 0.0614, Test RMSE: 0.2131


In [49]:
joblib.dump(ensemble,'../models/mustard.pkl')

['../models/mustard.pkl']

## Predicting the environment parameters of year 2023 using Prophet TSA model

In [42]:
# --- 1. Define function to forecast any single feature using Prophet ---
def forecast_feature_prophet(df, feature_name):
    forecast_data = []

    for (state, crop), group in tqdm(df.groupby(['state_name', 'crop_name'])):
        yearly_data = group.groupby('year')[feature_name].mean().reset_index()

        if yearly_data.shape[0] < 4:
            continue

        prophet_df = yearly_data.rename(columns={'year': 'ds', feature_name: 'y'})
        prophet_df['ds'] = pd.to_datetime(prophet_df['ds'], format='%Y')

        try:
            model = Prophet()
            model.fit(prophet_df)

            future = pd.DataFrame({'ds': [pd.to_datetime('2023')]})
            forecast = model.predict(future)
            yhat = forecast['yhat'].values[0]

            forecast_data.append({
                'state_name': state,
                'crop_name': crop,
                feature_name: yhat
            })
        except:
            continue

    return pd.DataFrame(forecast_data)

# --- 2. Forecast each feature separately ---
df_rain = forecast_feature_prophet(df, 'state_rainfall_val')
df_temp_max = forecast_feature_prophet(df, 'state_temperature_max_val')
df_temp_min = forecast_feature_prophet(df, 'state_temperature_min_val')
df_livecap = forecast_feature_prophet(df, 'Live Cap FRL')
df_frl = forecast_feature_prophet(df, 'FRL')
df_level = forecast_feature_prophet(df, 'Level')
df_cls = forecast_feature_prophet(df, 'Current Live Storage')

# --- 3. Merge all forecasted dataframes ---
from functools import reduce
dfs = [df_rain, df_temp_max, df_temp_min, df_livecap, df_frl, df_level, df_cls]
df_2023 = reduce(lambda left, right: pd.merge(left, right, on=['state_name', 'crop_name'], how='outer'), dfs)

# --- 4. One-hot encode state_name ---
df_2023_encoded = df_2023.copy()  # Keep original columns
state_names = df_2023_encoded[['state_name', 'crop_name']]  # Keep for merging later

df_2023_encoded = pd.get_dummies(df_2023_encoded, columns=['state_name'])
df_2023_encoded = pd.concat([state_names, df_2023_encoded.drop(columns=['crop_name'])], axis=1)


  0%|          | 0/12 [00:00<?, ?it/s]01:14:58 - cmdstanpy - INFO - Chain [1] start processing
01:14:58 - cmdstanpy - INFO - Chain [1] done processing
  8%|▊         | 1/12 [00:00<00:03,  3.66it/s]01:14:58 - cmdstanpy - INFO - Chain [1] start processing
01:14:58 - cmdstanpy - INFO - Chain [1] done processing
 17%|█▋        | 2/12 [00:00<00:02,  3.99it/s]01:14:59 - cmdstanpy - INFO - Chain [1] start processing
01:14:59 - cmdstanpy - INFO - Chain [1] done processing
 25%|██▌       | 3/12 [00:00<00:02,  3.92it/s]01:14:59 - cmdstanpy - INFO - Chain [1] start processing
01:14:59 - cmdstanpy - INFO - Chain [1] done processing
 33%|███▎      | 4/12 [00:01<00:01,  4.01it/s]01:14:59 - cmdstanpy - INFO - Chain [1] start processing
01:14:59 - cmdstanpy - INFO - Chain [1] done processing
 42%|████▏     | 5/12 [00:01<00:01,  3.69it/s]01:14:59 - cmdstanpy - INFO - Chain [1] start processing
01:14:59 - cmdstanpy - INFO - Chain [1] done processing
 50%|█████     | 6/12 [00:01<00:01,  3.95it/s]01:15:00

In [43]:
df_2023_encoded.head()

,state_name,crop_name,state_rainfall_val,state_temperature_max_val,state_temperature_min_val,Live Cap FRL,FRL,Level,Current Live Storage,state_name_Andhra Pradesh,...,state_name_Gujarat,state_name_Karnataka,state_name_Madhya Pradesh,state_name_Maharashtra,state_name_Odisha,state_name_Rajasthan,state_name_Tamil Nadu,state_name_Telangana,state_name_Uttar Pradesh,state_name_West Bengal
0,Andhra Pradesh,rapeseed &mustard,2.783452,34.311185,19.343944,2.177687,198.385252,142.288901,0.985749,True,...,False,False,False,False,False,False,False,False,False,False
1,Chhattisgarh,rapeseed &mustard,3.704114,33.805702,18.200942,1.365667,377.820000,353.377518,1.355013,False,...,False,False,False,False,False,False,False,False,False,False
2,Gujarat,rapeseed &mustard,2.706400,34.828181,18.653730,0.811491,115.199781,119.286907,0.366662,False,...,True,False,False,False,False,False,False,False,False,False
3,Karnataka,rapeseed &mustard,4.390836,32.315709,16.442800,1.539062,597.731875,587.411163,0.803126,False,...,False,True,False,False,False,False,False,False,False,False
4,Madhya Pradesh,rapeseed &mustard,3.183129,35.390574,17.187509,2.744818,373.365455,329.366944,1.735598,False,...,False,False,True,False,False,False,False,False,False,False


In [44]:
X_2023 = df_2023_encoded[features]

# Use your trained ensemble model
y_2023_pred = ensemble.predict(X_2023)

# Add prediction to the dataframe
df_2023_encoded['predicted_yield'] = y_2023_pred

# Select output
output_2023 = df_2023_encoded[['crop_name'] + [col for col in df_2023_encoded.columns if col.startswith('state_name_')] + ['predicted_yield']]


In [ ]:
# Convert dummy columns back to state_name
state_names = df_2023_encoded[[col for col in df_2023_encoded.columns if col.startswith('state_name_')]].idxmax(axis=1)
state_names = state_names.str.replace('state_name_', '')

# Final output
final_2023_yield = pd.DataFrame({
    'state_name': state_names,
    'crop_name': df_2023_encoded['crop_name'],
    'predicted_yield_2023': df_2023_encoded['predicted_yield'].round(2)
})

print(final_2023_yield)
# final_2023_yield.to_csv("../yield_prediction.csv", mode='a', header=False, index=False)

        state_name          crop_name  predicted_yield_2023
0   Andhra Pradesh  rapeseed &mustard                  0.37
1     Chhattisgarh  rapeseed &mustard                  0.34
2          Gujarat  rapeseed &mustard                  1.45
3        Karnataka  rapeseed &mustard                  0.30
4   Madhya Pradesh  rapeseed &mustard                  1.46
5      Maharashtra  rapeseed &mustard                  0.33
6           Odisha  rapeseed &mustard                  0.21
7        Rajasthan  rapeseed &mustard                  1.15
8       Tamil Nadu  rapeseed &mustard                  0.25
9        Telangana  rapeseed &mustard                  1.36
10   Uttar Pradesh  rapeseed &mustard                  0.99
11     West Bengal  rapeseed &mustard                  1.08


Comparing our predicted yield with previous year yields

In [46]:
# Step 1: Get actual yields from 2019 to 2022
df_recent = df_annual[df_annual['year'].between(2019, 2022)].copy()

# Pivot to get each year's yield as a column
yield_table = df_recent.pivot_table(
    index=['state_name', 'crop_name'],
    columns='year',
    values='yield'
).reset_index()

# Rename columns for clarity
yield_table = yield_table.rename(columns={
    2019: 'yield_2019',
    2020: 'yield_2020',
    2021: 'yield_2021',
    2022: 'yield_2022'
})

# Step 2: Prepare 2023 predicted yield
df_2023_yield = df_2023_encoded[['state_name', 'crop_name', 'predicted_yield']].copy()
df_2023_yield = df_2023_yield.rename(columns={'predicted_yield': 'yield_2023'})

# Step 3: Merge the 2023 predicted yield into the table
final_yield_table = pd.merge(yield_table, df_2023_yield, on=['state_name', 'crop_name'], how='left')

# Display final table
print(final_yield_table)


       state_name          crop_name  yield_2019  yield_2020  yield_2021  \
0  Andhra Pradesh  rapeseed &mustard     0.48459     0.64067     0.64458   
1    Chhattisgarh  rapeseed &mustard     0.44801     0.51701     0.51425   
2         Gujarat  rapeseed &mustard     1.93224     1.97572     1.99568   
3  Madhya Pradesh  rapeseed &mustard     1.53812     1.74513     1.37924   
4     Maharashtra  rapeseed &mustard     0.30465         NaN         NaN   
5       Rajasthan  rapeseed &mustard     1.58106     1.71250     1.71613   
6      Tamil Nadu  rapeseed &mustard     0.24088     0.23129     0.23571   
7   Uttar Pradesh  rapeseed &mustard     1.26038     1.43761     1.36776   
8     West Bengal  rapeseed &mustard     1.16705     1.24964     1.21822   

   yield_2022  yield_2023  
0     0.74224    0.369962  
1     0.56299    0.338450  
2     1.96648    1.445962  
3     1.54036    1.456797  
4         NaN    0.325325  
5     1.46835    1.147630  
6     0.23823    0.253244  
7     1.49731  